In [1]:
# Import packages
import matplotlib.pyplot as plt
import osgeo
from os.path import join as pjoin
import pandas as pd
import numpy as np
import xarray as xr
import xrspatial as xrs
import rioxarray

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas as gpd
import seaborn as sns
import datashader as ds

import pystac
from pystac_client import Client
import planetary_computer
import requests
import stackstac
from glob import glob

import rich.table
import dask.diagnostics

path = os.getcwd()
parent = os.sep.join(path.split(os.sep)[:-1])

YARA INDEXATION ON GMM AND K-MEANS (GRAPHS)

In [2]:
#lOAD DATA
import os
import glob
import pandas as pd
import xarray as xr
import rioxarray

repo_dir = r"C:/Users/yaroc/OneDrive/Документы/GitHub/RemoteSensing_CoralReefs"

pattern1 = os.path.join(repo_dir, "data", "raw", "PSScene", "*_3B_AnalyticMS_SR_8b_clip_philip.tif")
pattern2 = os.path.join(repo_dir, "data", "raw", "PSScene", "*_3B_AnalyticMS_SR_8b_clip_tricia.tif")
pattern3 = os.path.join(repo_dir, "data", "raw", "PSScene", "*_3B_AnalyticMS_SR_8b_clip_dennis.tif")

files = sorted(glob.glob(pattern1, recursive=True) + glob.glob(pattern2, recursive=True) + glob.glob(pattern3, recursive=True))

dates = [pd.Timestamp(os.path.basename(f)[:8]) for f in files]

arrays = []
for f, date in zip(files, dates):
    da = rioxarray.open_rasterio(
        f,
        chunks={"band": 1, "x": 512, "y": 512}
    )
    da = da.assign_coords(time=date)
    arrays.append(da)

stack = xr.concat(arrays, dim="time").sortby("time")

print(stack)

C:\Windows\Temp\ipykernel_5776\1084959129.py:27: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'x' ('x',) The recommendation is to set join explicitly for this case.
  stack = xr.concat(arrays, dim="time").sortby("time")
C:\Windows\Temp\ipykernel_5776\1084959129.py:27: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'y' ('y',) The recommendation is to set join explicitly for this case.
  stack = xr.concat(arrays, dim="time").sortby("time")


<xarray.DataArray (time: 106, band: 8, y: 1237, x: 2030)> Size: 9GB
dask.array<getitem, shape=(106, 8, 1237, 2030), dtype=float32, chunksize=(1, 1, 165, 351), chunktype=numpy.ndarray>
Coordinates:
  * time         (time) datetime64[ns] 848B 2022-01-08 2022-01-17 ... 2025-12-10
  * band         (band) int32 32B 1 2 3 4 5 6 7 8
  * y            (y) float64 10kB 7.399e+06 7.399e+06 ... 7.403e+06 7.403e+06
  * x            (x) float64 16kB 4.019e+05 4.019e+05 ... 4.08e+05 4.08e+05
    spatial_ref  int32 4B 0
Attributes:
    AREA_OR_POINT:             Area
    TIFFTAG_DATETIME:          2022:01:08 23:09:32
    TIFFTAG_IMAGEDESCRIPTION:  {"atmospheric_correction": {"aerosol_model": "...
    _FillValue:                0
    scale_factor:              1.0
    add_offset:                0.0
    long_name:                 ('coastal_blue', 'blue', 'green_i', 'green', '...


In [3]:
import time
import xarray as xr
import numpy as np
import pandas as pd

t0 = time.time()

# 1. Select bands
coastal_blue = stack.isel(band=0, drop=True)
blue         = stack.isel(band=1, drop=True)
green        = stack.isel(band=3, drop=True)
red          = stack.isel(band=5, drop=True)
nir          = stack.isel(band=7, drop=True)
yellow       = stack.isel(band=4, drop=True)

# 2. Visible
visible = (coastal_blue + blue + green + red + yellow) / 5

# 3. Safe divide
def safe_divide(a, b):
    return xr.where(b != 0, a / b, np.nan)

# 4. Indices
blue_green = safe_divide(coastal_blue, green)
brightness = visible
nbrci      = safe_divide((coastal_blue - nir), (coastal_blue + nir))
ndavi      = safe_divide((nir - coastal_blue), (nir + coastal_blue))
t1 = time.time()
print(f"Index computation: {t1 - t0:.2f} sec")

# 5. Means
blue_green_mean = blue_green.fillna(0).mean(dim=("y", "x"))
brightness_mean = brightness.fillna(0).mean(dim=("y", "x"))
nbrci_mean      = nbrci.fillna(0).mean(dim=("y", "x"))
ndavi_mean      = ndavi.fillna(0).mean(dim=("y", "x"))
t2 = time.time()
print(f"Spatial mean: {t2 - t1:.2f} sec")

# 6. Dataset + dataframe
ds_indexes = xr.Dataset({
    "blue_green": blue_green_mean,
    "brightness": brightness_mean,
    "nbrci": nbrci_mean,
    "ndavi": ndavi_mean
})

df_indexes = ds_indexes.to_dataframe().reset_index()
df_indexes["Date"] = pd.to_datetime(df_indexes["time"]).dt.strftime("%Y-%m-%d")
df_indexes = df_indexes.drop(columns=["time"])
t3 = time.time()
print(f"Conversion to dataframe: {t3 - t2:.2f} sec")

print(f"Total runtime: {t3 - t0:.2f} sec")
#save to csv
csv_path = os.path.join(repo_dir, "data", "PSScene", "coral_bleaching_indices.csv")  
df_indexes.to_csv(csv_path, index=False)

Index computation: 2.15 sec
Spatial mean: 6.98 sec


KeyboardInterrupt: 

In [ ]:
df_indexes.head()

In [ ]:
# Copy to avoid overwriting original values
df_numeric = df_indexes.drop(columns=["Date"])
df_norm = (df_numeric - df_numeric.mean()) / df_numeric.std()

df_norm["Date"] = df_indexes["Date"]